# SPlus / Schedule-Free / AdamW на IWSLT14 De→En (Transformer)
Порядок: настройки → данные → модель → оптимизаторы → функции обучения → TensorBoard → запуск → сводка.
`SMOKE = True` — быстрая проверка пайплайна (если папки с данными нет, берётся игрушечная задача «развернуть последовательность»).
`MODEL_SIZE = 'big'` (~37M) или `'small'` (~14M). Результаты: `custom_experiment_results/iwslt14_<размер>/`.

In [1]:
import collections, glob, json, math, os, random, time
import torch, torch.nn as nn, torch.nn.functional as F
import sacrebleu
from torch.utils.tensorboard import SummaryWriter
import schedulefree
from schedulefree import ScheduleFreeWrapper
from splus import SPlus

In [3]:
SMOKE = False          # True - быстрая проверка, False - полный эксперимент
MODEL_SIZE = 'big'    # 'big' (~37M параметров) или 'small' (~14M); в режиме SMOKE игнорируется
METHODS_TO_RUN = ['adamw', 'adamw_sf', 'splus', 'splus_sf']
SEEDS = [0]
DATA_DIR = 'iwslt14.tokenized.de-en'

OUT = 'iwlst14_exp_resulsts/iwslt14_' + ('smoke' if SMOKE else MODEL_SIZE)
EPOCHS        = 2    if SMOKE else 50
MAX_TOKENS    = 512  if SMOKE else 4096      # токенов в батче (рецепт fairseq)
WARMUP        = 20   if SMOKE else 4000      # шагов
INVERSE_EVERY = 10   if SMOKE else 100       # как часто пересчитывать собственные базисы SPlus
LOG_EVERY     = 10   if SMOKE else 50        # как часто писать лосс по шагам в tensorboard
DROPOUT, LABEL_SMOOTHING, MAX_LEN = 0.3, 0.1, 256

ARCH = {'big':   dict(d=512, ffn=1024, heads=4, layers=6),
        'small': dict(d=256, ffn=1024, heads=4, layers=6),
        'smoke': dict(d=64,  ffn=128,  heads=4, layers=2)}['smoke' if SMOKE else MODEL_SIZE]

LR = {  # типичный рецепт fairseq для adam; для SPlus около 0.2 (как в статье) - подберите короткой пробой
    'adamw': 5e-4, 'adamw_sf': 5e-4, 'splus': 0.2, 'splus_sf': 0.2,
}
# sched: 'invsqrt' (стандарт для этой модели) | 'const' (после warmup). evals: где оценивать ([] = живые веса,
# первая точка основная): 'sf' - точка Schedule-Free, 'ema' - EMA-веса SPlus
METHODS = {
    'adamw':    dict(wd=1e-4, sched='invsqrt', evals=[]),
    'adamw_sf': dict(wd=1e-4, sched='const',   evals=['sf']),
    'splus':    dict(wd=0.1,  sched='const',   evals=['ema']),
    'splus_sf': dict(wd=0.1,  sched='const',   evals=['sf']),   # SF + EMA: логируем обе точки
}
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(OUT, exist_ok=True)
print('device:', dev, '| results ->', OUT)

device: cpu | results -> iwlst14_exp_resulsts/iwslt14_big


**Данные (один раз).** Нужны `git`, `perl`, `wget`, `python`. Скрипт fairseq скачивает IWSLT14, токенизирует, переводит в нижний регистр и режет на BPE (10K);
на выходе папка `iwslt14.tokenized.de-en` с файлами `train/valid/test.{de,en}`. Раскомментируйте и выполните:

In [4]:
PAD, BOS, EOS, UNK = 0, 1, 2, 3

def load_split(split):
    n = {'train': 2000, 'valid': 200, 'test': 100}[split]
    if os.path.isdir(DATA_DIR):
        rd = lambda l: [ln.split() for ln in open(f'{DATA_DIR}/{split}.{l}', encoding='utf-8')]
        de, en = rd('de'), rd('en')
        return (de[:n], en[:n]) if SMOKE else (de, en)
    assert SMOKE, f'нет папки {DATA_DIR}: подготовьте данные (ячейка выше)'
    rnd = random.Random({'train': 0, 'valid': 1, 'test': 2}[split])       # игрушечная задача для проверки кода
    de = [[f'w{rnd.randrange(200)}' for _ in range(rnd.randint(3, 12))] for _ in range(n)]
    return de, [list(reversed(s)) for s in de]

(tr_de, tr_en), (va_de, va_en), (te_de, te_en) = load_split('train'), load_split('valid'), load_split('test')
counter = collections.Counter(t for s in tr_de + tr_en for t in s)
itos = ['<pad>', '<s>', '</s>', '<unk>'] + [t for t, _ in counter.most_common()]     # общий словарь de+en
stoi = {t: i for i, t in enumerate(itos)}
V = len(itos)

def encode(de, en, max_len=None):
    pairs = []
    for s, t in zip(de, en):
        s_ids, t_ids = [stoi.get(w, UNK) for w in s] + [EOS], [stoi.get(w, UNK) for w in t]
        if max_len is None or (len(s_ids) <= max_len and len(t_ids) + 1 <= max_len):
            pairs.append((s_ids, t_ids))
    return pairs

def make_batches(pairs, max_tokens):          # батчи из предложений близкой длины, не более max_tokens токенов
    idx = sorted(range(len(pairs)), key=lambda i: (len(pairs[i][1]), len(pairs[i][0])))
    batches, cur, mx = [], [], 0
    for i in idx:
        L = max(len(pairs[i][0]), len(pairs[i][1]) + 1)
        if cur and max(mx, L) * (len(cur) + 1) > max_tokens:
            batches.append(cur); cur, mx = [], 0
        cur.append(i); mx = max(mx, L)
    return batches + ([cur] if cur else [])

def to_tensors(pairs, batches):               # все батчи один раз кладём на GPU
    out = []
    for b in batches:
        S, T = max(len(pairs[i][0]) for i in b), max(len(pairs[i][1]) for i in b) + 1
        src, tin, tout = (torch.full((len(b), n), PAD) for n in (S, T, T))
        for r, i in enumerate(b):
            s, t = pairs[i]
            src[r, :len(s)] = torch.tensor(s)
            tin[r, :len(t) + 1] = torch.tensor([BOS] + t)
            tout[r, :len(t) + 1] = torch.tensor(t + [EOS])
        out.append((src.to(dev), tin.to(dev), tout.to(dev), sum(len(pairs[i][1]) + 1 for i in b)))
    return out

train_pairs = encode(tr_de, tr_en, MAX_LEN)
valid_pairs, test_pairs = encode(va_de, va_en), encode(te_de, te_en)
train_batches = to_tensors(train_pairs, make_batches(train_pairs, MAX_TOKENS))
valid_batches = to_tensors(valid_pairs, make_batches(valid_pairs, MAX_TOKENS))
test_idx = make_batches(test_pairs, MAX_TOKENS)
test_batches = to_tensors(test_pairs, test_idx)
test_refs = [' '.join(s).replace('@@ ', '') for s in te_en]
print(f'vocab {V} | train {len(train_pairs)} пар, {len(train_batches)} батчей/эпоху | valid {len(valid_pairs)} | test {len(test_pairs)}')

vocab 10148 | train 160239 пар, 1073 батчей/эпоху | valid 7283 | test 6750


In [5]:
def sinusoid(max_len, d):
    ang = torch.arange(max_len)[:, None] / (10000 ** (torch.arange(0, d, 2)[None] / d))
    pe = torch.zeros(max_len, d)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(ang), torch.cos(ang)
    return pe


class Seq2Seq(nn.Module):
    # pre-LN трансформер; один общий эмбеддинг для энкодера, декодера и выходного слоя
    def __init__(self, V, d, ffn, heads, layers, dropout):
        super().__init__()
        self.d = d
        self.emb = nn.Embedding(V, d, padding_idx=PAD)
        self.register_buffer('pe', sinusoid(1024, d), persistent=False)
        self.drop = nn.Dropout(dropout)
        enc = nn.TransformerEncoderLayer(d, heads, ffn, dropout, batch_first=True, norm_first=True)
        dec = nn.TransformerDecoderLayer(d, heads, ffn, dropout, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(enc, layers, norm=nn.LayerNorm(d), enable_nested_tensor=False)
        self.dec = nn.TransformerDecoder(dec, layers, norm=nn.LayerNorm(d))
        for n, p in self.named_parameters():
            if p.dim() > 1 and n != 'emb.weight':
                nn.init.xavier_uniform_(p)
        nn.init.normal_(self.emb.weight, 0, d ** -0.5)
        with torch.no_grad():
            self.emb.weight[PAD].zero_()

    def embed(self, x):
        return self.drop(self.emb(x) * math.sqrt(self.d) + self.pe[:x.size(1)])

    def encode(self, src):
        return self.enc(self.embed(src), src_key_padding_mask=src.eq(PAD))

    def decode(self, tin, mem, src):
        T = tin.size(1)
        causal = torch.triu(torch.ones(T, T, dtype=torch.bool, device=tin.device), 1)
        h = self.dec(self.embed(tin), mem, tgt_mask=causal, memory_key_padding_mask=src.eq(PAD))
        return h @ self.emb.weight.T

    def forward(self, src, tin):
        return self.decode(tin, self.encode(src), src)


def make_model():
    return Seq2Seq(V, dropout=DROPOUT, **ARCH)

print(f'params: {sum(p.numel() for p in make_model().parameters()) / 1e6:.2f}M  {ARCH}')

params: 36.74M  {'d': 512, 'ffn': 1024, 'heads': 4, 'layers': 6}


In [5]:
class SFWrapper(ScheduleFreeWrapper):
    # ScheduleFreeWrapper кладёт свой буфер 'z' в base.state[p], а SPlus считает состояние неинициализированным
    # только пока len(state) == 0. Чтобы они не мешали друг другу, храним состояние обёртки отдельно.
    def __init__(self, *a, **k):
        super().__init__(*a, **k)
        self._sf_state = collections.defaultdict(dict)

    @property
    def state(self):
        return self._sf_state


def build(name, model, lr):
    m = METHODS[name]
    if name == 'adamw':
        opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.98), weight_decay=m['wd'])
    elif name == 'adamw_sf':   # warmup делаем сами через lr_factor
        opt = schedulefree.AdamWScheduleFree(model.parameters(), lr=lr, betas=(0.9, 0.98), weight_decay=m['wd'])
    else:
        emb = [model.emb.weight]
        body = [p for p in model.parameters() if p is not model.emb.weight]
        sf = name.endswith('_sf')
        # эмбеддинг (он же выходной слой): max_dim=0 -> без собственного базиса, обновление sign(momentum) (как в статье)
        base = SPlus([{'params': body}, {'params': emb, 'max_dim': 0}], lr=lr,
                     b1=0.0 if sf else 0.9,            # в SF-вариантах импульс даёт обёртка
                     weight_decay=m['wd'], inverse_every=INVERSE_EVERY)
        opt = SFWrapper(base, momentum=0.9) if sf else base
    for g in opt.param_groups:
        g['base_lr'] = g['lr']
    if 'sf' in m['evals']:
        opt.train()                                     # SF-оптимизаторы требуют train() до первого шага
    return opt


def lr_factor(step, sched):
    w = (step + 1) / WARMUP
    return min(w, math.sqrt(WARMUP / (step + 1))) if sched == 'invsqrt' else min(1.0, w)

In [6]:
AMP = dev.type == 'cuda'

@torch.no_grad()
def evaluate(model, batches):                 # NLL на токен (без label smoothing) и точность по токенам
    model.eval()
    nll = correct = n = 0
    for src, tin, tout, ntok in batches:
        with torch.autocast(dev.type, torch.bfloat16, enabled=AMP):
            logits = model(src, tin)
        lg, t = logits.float().view(-1, V), tout.view(-1)
        nll += F.cross_entropy(lg, t, ignore_index=PAD, reduction='sum').item()
        correct += ((lg.argmax(-1) == t) & (t != PAD)).sum().item()
        n += ntok
    return nll / n, correct / n


@torch.no_grad()
def greedy_decode(model, src):
    mem, ys = model.encode(src), torch.full((src.size(0), 1), BOS, device=src.device)
    done = torch.zeros(src.size(0), dtype=torch.bool, device=src.device)
    for _ in range(int(1.5 * src.size(1)) + 10):
        nxt = model.decode(ys, mem, src)[:, -1].argmax(-1).masked_fill(done, PAD)
        ys = torch.cat([ys, nxt[:, None]], 1)
        done |= nxt.eq(EOS)
        if done.all():
            break
    return ys[:, 1:]


def to_text(ids):
    out = []
    for i in ids.tolist():
        if i == EOS:
            break
        out.append(itos[i])
    return ' '.join(out).replace('@@ ', '')


@torch.no_grad()
def bleu(model):                              # BLEU на токенизированном lowercase-тексте, greedy-декодирование
    model.eval()
    hyps, refs = [], []
    for (src, *_), idxs in zip(test_batches, test_idx):
        with torch.autocast(dev.type, torch.bfloat16, enabled=AMP):
            ys = greedy_decode(model, src)
        hyps += [to_text(y) for y in ys]
        refs += [test_refs[i] for i in idxs]
    return sacrebleu.corpus_bleu(hyps, [refs], tokenize='none').score


def with_point(opt, point, fn):
    if point is None:                          # живые веса
        return fn()
    o = opt if point == 'sf' else getattr(opt, 'base', opt)    # 'ema' -> сам SPlus (внутри обёртки, если она есть)
    o.eval()
    try:
        return fn()
    finally:
        o.train()


def run(name, seed):
    lr, m = LR[name], METHODS[name]
    tag = f'{name}_lr{lr:.4g}_s{seed}'
    path = f'{OUT}/{tag}.json'
    if os.path.exists(path):
        print('skip', tag)
        return
    torch.manual_seed(seed)                    # одинаковая инициализация у всех методов при одном seed
    rng = random.Random(seed)
    model = make_model().to(dev)
    opt = build(name, model, lr)
    points = m['evals'] or [None]

    log = dict(cfg=dict(method=name, lr=lr, seed=seed, epochs=EPOCHS, max_tokens=MAX_TOKENS,
                        inverse_every=INVERSE_EVERY, arch=ARCH, **m),
               train_loss=[], valid_nll=[], valid_ppl=[], valid_acc=[])
    if len(points) > 1:
        log.update(valid_nll_ema=[], valid_acc_ema=[])
    writer = SummaryWriter(f'{OUT}/tb/{tag}')
    step, t0, diverged = 0, time.time(), False

    for ep in range(EPOCHS):
        model.train()
        order = list(range(len(train_batches)))
        rng.shuffle(order)
        sum_loss, n_tok = torch.zeros((), device=dev), 0
        for bi in order:
            src, tin, tout, ntok = train_batches[bi]
            f = lr_factor(step, m['sched'])
            for g in opt.param_groups:
                g['lr'] = g['base_lr'] * f
            opt.zero_grad(set_to_none=True)
            with torch.autocast(dev.type, torch.bfloat16, enabled=AMP):
                logits = model(src, tin)
            loss = F.cross_entropy(logits.float().view(-1, V), tout.view(-1), ignore_index=PAD,
                                   label_smoothing=LABEL_SMOOTHING)
            loss.backward()
            opt.step()
            sum_loss += loss.detach() * ntok
            n_tok += ntok
            step += 1
            if step % LOG_EVERY == 0:          # .item() синхронизирует GPU, поэтому не на каждом шаге
                v = loss.item()
                writer.add_scalar('step/train_loss', v, step)
                writer.add_scalar('step/lr_factor', f, step)
                if not math.isfinite(v):
                    diverged = True
                    break
        if diverged:
            print(f'{tag}: DIVERGED at step {step}')
            break

        res = [with_point(opt, p, lambda: evaluate(model, valid_batches)) for p in points]
        tr = sum_loss.item() / n_tok
        log['train_loss'].append(tr)
        log['valid_nll'].append(res[0][0]); log['valid_ppl'].append(math.exp(res[0][0])); log['valid_acc'].append(res[0][1])
        writer.add_scalar('epoch/train_loss', tr, ep)
        writer.add_scalar('epoch/valid_nll', res[0][0], ep)
        writer.add_scalar('epoch/valid_ppl', math.exp(res[0][0]), ep)
        writer.add_scalar('epoch/valid_acc', res[0][1], ep)
        if len(res) > 1:
            log['valid_nll_ema'].append(res[1][0]); log['valid_acc_ema'].append(res[1][1])
            writer.add_scalar('epoch/valid_nll_ema', res[1][0], ep)
        if (ep + 1) % (1 if SMOKE else 5) == 0 or ep == EPOCHS - 1:
            extra = f' | ema nll {res[1][0]:.3f}' if len(res) > 1 else ''
            print(f'{tag} ep {ep + 1}/{EPOCHS} train {tr:.3f} valid nll {res[0][0]:.3f} ppl {math.exp(res[0][0]):.1f} '
                  f'acc {res[0][1]:.4f}{extra} | {time.time() - t0:.0f}s', flush=True)

    if not diverged:                            # BLEU финальной модели (в каждой точке оценки)
        scores = [with_point(opt, p, lambda: bleu(model)) for p in points]
        log['bleu'] = scores[0]
        if len(scores) > 1:
            log['bleu_ema'] = scores[1]
        writer.add_scalar('final/bleu', scores[0], 0)
        print(f'{tag} BLEU (greedy): {scores}')
    log['time'], log['diverged'] = time.time() - t0, diverged
    writer.close()
    json.dump(log, open(path, 'w'))
    torch.cuda.empty_cache()

Запустите эту ячейку **до** обучения: TensorBoard откроется прямо в ноутбуке и будет обновляться во время запусков.

In [7]:
# ---------------- запуск эксперимента ----------------
for seed in SEEDS:                     # seed снаружи: после первого seed уже есть сравнение всех методов
    for name in METHODS_TO_RUN:
        run(name, seed)

adamw_lr0.0005_s0 ep 1/2 train 8.209 valid nll 7.116 ppl 1231.0 acc 0.0495 | 2s
adamw_lr0.0005_s0 ep 2/2 train 7.174 valid nll 6.647 ppl 770.7 acc 0.0484 | 4s
adamw_lr0.0005_s0 BLEU (greedy): [0.015772559412096904]
adamw_sf_lr0.0005_s0 ep 1/2 train 8.283 valid nll 7.178 ppl 1309.7 acc 0.0495 | 2s
adamw_sf_lr0.0005_s0 ep 2/2 train 7.069 valid nll 6.566 ppl 710.4 acc 0.0437 | 4s
adamw_sf_lr0.0005_s0 BLEU (greedy): [0.015772559412096904]
splus_lr0.2_s0 ep 1/2 train 8.220 valid nll 7.706 ppl 2221.1 acc 0.0979 | 2s
splus_lr0.2_s0 ep 2/2 train 6.908 valid nll 6.893 ppl 984.9 acc 0.1130 | 4s
splus_lr0.2_s0 BLEU (greedy): [0.03398423892628086]
splus_sf_lr0.2_s0 ep 1/2 train 8.805 valid nll 8.080 ppl 3229.2 acc 0.0495 | 2s
splus_sf_lr0.2_s0 ep 2/2 train 7.977 valid nll 7.455 ppl 1728.9 acc 0.0680 | 5s
splus_sf_lr0.2_s0 BLEU (greedy): [0.015772559412096904]


In [8]:
# ---------------- сводка ----------------
for f in sorted(glob.glob(f'{OUT}/*.json')):
    r = json.load(open(f))
    c = r['cfg']
    if r['diverged'] or not r['valid_nll']:
        print(f"{c['method']:10s} seed {c['seed']}  lr {c['lr']:<8g} DIVERGED")
        continue
    ema = f" | ema nll {r['valid_nll_ema'][-1]:.3f}, BLEU {r['bleu_ema']:.2f}" if 'bleu_ema' in r else ''
    print(f"{c['method']:10s} seed {c['seed']}  lr {c['lr']:<8g} valid nll {r['valid_nll'][-1]:.3f} "
          f"ppl {r['valid_ppl'][-1]:.2f}  BLEU {r['bleu']:.2f}{ema}  ({r['time']:.0f}s)")

adamw      seed 0  lr 0.0005   valid nll 6.647 ppl 770.73  BLEU 0.02  (6s)
adamw_sf   seed 0  lr 0.0005   valid nll 6.566 ppl 710.40  BLEU 0.02  (6s)
splus      seed 0  lr 0.2      valid nll 6.893 ppl 984.90  BLEU 0.03  (7s)
splus_sf   seed 0  lr 0.2      valid nll 7.455 ppl 1728.92  BLEU 0.02  (7s)
